In [1]:
from collections import defaultdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_ID = "roneneldan/TinyStories-3M"

# Pin the exact model revision for reproducible experiments.
MODEL_REVISION = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

W0923 17:06:32.859000 9466 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0923 17:06:32.889000 9466 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 29149.60it/s]
[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-3M
Key                                                               | Status     |  | 
------------------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.att

In [4]:
model.eval()

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 128)
    (wpe): Embedding(2048, 128)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-7): 8 x GPTNeoBlock(
        (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=128, out_features=128, bias=False)
            (v_proj): Linear(in_features=128, out_features=128, bias=False)
            (q_proj): Linear(in_features=128, out_features=128, bias=False)
            (out_proj): Linear(in_features=128, out_features=128, bias=True)
          )
        )
        (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=128, out_features=512, bias=True)
          (c_

In [5]:
model.transformer.h[0]

GPTNeoBlock(
  (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (attn): GPTNeoAttention(
    (attention): GPTNeoSelfAttention(
      (attn_dropout): Dropout(p=0.0, inplace=False)
      (resid_dropout): Dropout(p=0.0, inplace=False)
      (k_proj): Linear(in_features=128, out_features=128, bias=False)
      (v_proj): Linear(in_features=128, out_features=128, bias=False)
      (q_proj): Linear(in_features=128, out_features=128, bias=False)
      (out_proj): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (mlp): GPTNeoMLP(
    (c_fc): Linear(in_features=128, out_features=512, bias=True)
    (c_proj): Linear(in_features=512, out_features=128, bias=True)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.0, inplace=False)
  )
)

In [6]:
import numpy as np
from pathlib import Path
from torch import nn

try:
    import litert_torch
    print("LiteRT Torch version:", litert_torch.__version__)
except ImportError:
    print("LiteRT Torch is not installed.")

I0000 00:00:1790154394.607266    9466 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


LiteRT Torch version: 0.9.4


In [7]:
SEQUENCE_LENGTH = 32

prompt = (
    "Once upon a time, a curious little girl named Lily lived in a "
    "quiet village beside a green forest. Every morning, she carried "
    "fresh bread to her grandmother and watched the birds in the trees."
)

encoded = tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
)

available_tokens = encoded["input_ids"].shape[1]
print("Available prompt tokens:", available_tokens)

assert available_tokens >= SEQUENCE_LENGTH

input_ids = encoded["input_ids"][:, :SEQUENCE_LENGTH]

position_ids = torch.arange(
    SEQUENCE_LENGTH,
    dtype=torch.long,
).unsqueeze(0)

with torch.no_grad():
    token_embeddings = model.transformer.wte(input_ids)
    position_embeddings = model.transformer.wpe(position_ids)

    block_input = model.transformer.drop(
        token_embeddings + position_embeddings
    )

print("Input IDs shape:", input_ids.shape)
print("Block input shape:", block_input.shape)
print("Block input dtype:", block_input.dtype)
print(
    "Block input range:",
    block_input.min().item(),
    block_input.max().item(),
)

Available prompt tokens: 39
Input IDs shape: torch.Size([1, 32])
Block input shape: torch.Size([1, 32, 128])
Block input dtype: torch.float32
Block input range: -0.3287893235683441 0.30728602409362793


In [8]:
class Block0Wrapper(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.block = block

    def forward(self, hidden_states):
        outputs = self.block(
            hidden_states,
            layer_past=None,
            attention_mask=None,
            use_cache=False,
            output_attentions=False,
        )

        return outputs[0]


block0 = Block0Wrapper(model.transformer.h[0]).eval()

with torch.no_grad():
    pytorch_output = block0(block_input)

print("Output shape:", pytorch_output.shape)
print("Output dtype:", pytorch_output.dtype)
print(
    "Output range:",
    pytorch_output.min().item(),
    pytorch_output.max().item(),
)

Output shape: torch.Size([1, 32, 128])
Output dtype: torch.float32
Output range: -0.6481602787971497 0.5356095433235168


In [9]:
sample_inputs = (block_input,)

exported_program = torch.export.export(
    block0,
    sample_inputs,
)

print("torch.export succeeded")
print(
    "Exported graph nodes:",
    len(list(exported_program.graph.nodes)),
)

torch.export succeeded
Exported graph nodes: 62


In [10]:
edge_model = litert_torch.convert(
    block0,
    sample_inputs,
)

print("LiteRT conversion succeeded")

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default (+00:00)

... (15 outputs left out)

(00:01) [START] LiteRT-Torch Convert > Run LiteRT Converter Passes

(00:01) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:01) [ DONE] LiteRT-Torch Convert (+00:01)

LiteRT conversion succeeded


In [11]:
litert_output = edge_model(*sample_inputs)

if isinstance(litert_output, (tuple, list)):
    litert_output = litert_output[0]

litert_output = np.asarray(litert_output)
pytorch_output_np = pytorch_output.detach().cpu().numpy()

absolute_error = np.abs(
    pytorch_output_np - litert_output
)

print("LiteRT output shape:", litert_output.shape)
print("Maximum absolute error:", absolute_error.max())
print("Mean absolute error:", absolute_error.mean())

outputs_close = np.allclose(
    pytorch_output_np,
    litert_output,
    atol=1e-4,
    rtol=1e-4,
)

print("Outputs sufficiently close:", outputs_close)

(00:00) [START] Write Model to Bytes

I0000 00:00:1790154587.976979    9466 flatbuffer_export.cc:4342] Estimated count of arithmetic ops: 13.689 M  ops, equivalently 6.845 M  MACs


(00:00) [ DONE] Write Model to Bytes (+00:00)

LiteRT output shape: (1, 32, 128)
Maximum absolute error: 2.3841858e-07
Mean absolute error: 3.0009573e-08
Outputs sufficiently close: True


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
ERROR: Subgraph reshaping is enabled by default, TFLITE_XNNPACK_DELEGATE_FLAG_ENABLE_SUBGRAPH_RESHAPING is deprecated and will be removed in the future.


In [12]:
litert_output = edge_model(*sample_inputs)

if isinstance(litert_output, (tuple, list)):
    litert_output = litert_output[0]

litert_output = np.asarray(litert_output)
pytorch_output_np = pytorch_output.detach().cpu().numpy()

absolute_error = np.abs(
    pytorch_output_np - litert_output
)

print("LiteRT output shape:", litert_output.shape)
print("Maximum absolute error:", absolute_error.max())
print("Mean absolute error:", absolute_error.mean())

outputs_close = np.allclose(
    pytorch_output_np,
    litert_output,
    atol=1e-4,
    rtol=1e-4,
)

print("Outputs sufficiently close:", outputs_close)

LiteRT output shape: (1, 32, 128)
Maximum absolute error: 2.3841858e-07
Mean absolute error: 3.0009573e-08
Outputs sufficiently close: True


In [13]:
output_directory = Path("models")
output_directory.mkdir(exist_ok=True)

output_path = (
    output_directory
    / "tinystories_block0_fp32.tflite"
)

edge_model.export(str(output_path))

print("Saved model:", output_path.resolve())
print(
    "File size:",
    output_path.stat().st_size,
    "bytes",
)

Saved model: /home/randy/Documents/ACompArc/model/project/models/tinystories_block0_fp32.tflite
File size: 818532 bytes


In [14]:
from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e,
)

from litert_torch.quantize.pt2e_quantizer import (
    PT2EQuantizer,
    get_symmetric_quantization_config,
)

from litert_torch.quantize.quant_config import QuantConfig

In [15]:
quantization_config = get_symmetric_quantization_config(
    is_per_channel=False,
    is_dynamic=False,
    is_qat=False,
)

quantizer = PT2EQuantizer().set_global(
    quantization_config
)

print("Static W8A8 quantizer created")
print("Dynamic quantization:", quantization_config.is_dynamic)

Static W8A8 quantizer created
Dynamic quantization: False


In [16]:
quantized_exported_program = torch.export.export(
    block0,
    sample_inputs,
)

prepared_model = prepare_pt2e(
    quantized_exported_program.module(),
    quantizer,
)

print("PT2E preparation succeeded")

PT2E preparation succeeded


In [17]:
with torch.no_grad():
    calibration_output = prepared_model(
        block_input
    )

print("Calibration succeeded")
print("Calibration output shape:", calibration_output.shape)

Calibration succeeded
Calibration output shape: torch.Size([1, 32, 128])


In [18]:
quantized_model = convert_pt2e(
    prepared_model,
    fold_quantize=False,
)

print("PT2E quantization conversion succeeded")

PT2E quantization conversion succeeded


In [19]:
with torch.no_grad():
    quantized_torch_output = quantized_model(
        block_input
    )

print("Quantized output shape:",
      quantized_torch_output.shape)
print("Quantized output dtype:",
      quantized_torch_output.dtype)

Quantized output shape: torch.Size([1, 32, 128])
Quantized output dtype: torch.float32


In [20]:
fp32_output = pytorch_output.detach().cpu()
pt2e_output = quantized_torch_output.detach().cpu()

quantization_error = fp32_output - pt2e_output

mean_absolute_error = (
    quantization_error.abs().mean().item()
)

maximum_absolute_error = (
    quantization_error.abs().max().item()
)

cosine_similarity = torch.nn.functional.cosine_similarity(
    fp32_output.flatten(),
    pt2e_output.flatten(),
    dim=0,
).item()

print("FP32 versus PT2E W8A8")
print("Mean absolute error:", mean_absolute_error)
print("Maximum absolute error:", maximum_absolute_error)
print("Cosine similarity:", cosine_similarity)

FP32 versus PT2E W8A8
Mean absolute error: 0.001388444914482534
Maximum absolute error: 0.007108405232429504
Cosine similarity: 0.9998918175697327


In [21]:
quantization_nodes = []

for node in quantized_model.graph.nodes:
    target = str(node.target)

    if (
        "quantize" in target
        or "dequantize" in target
    ):
        quantization_nodes.append(
            (node.name, node.op, target)
        )

print(
    "Number of quantize/dequantize nodes:",
    len(quantization_nodes),
)

for name, operation, target in quantization_nodes:
    print(f"{name:45s} {operation:15s} {target}")

Number of quantize/dequantize nodes: 65
quantize_per_tensor_default                   call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default                 call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_1                 call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default_1               call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_2                 call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default_2               call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_3                 call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default_3               call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_4                 call_function   qu

In [22]:
for node in quantized_model.graph.nodes:
    target = str(node.target)

    if any(
        keyword in target
        for keyword in [
            "linear",
            "mm",
            "addmm",
            "matmul",
            "quantize",
            "dequantize",
        ]
    ):
        print(
            f"{node.name:45s} "
            f"{node.op:15s} "
            f"{target}"
        )

quantize_per_tensor_default                   call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default                 call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_1                 call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default_1               call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_2                 call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default_2               call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_3                 call_function   quantized_decomposed.quantize_per_tensor.default
dequantize_per_tensor_default_3               call_function   quantized_decomposed.dequantize_per_tensor.default
quantize_per_tensor_default_4                 call_function   quantized_decomposed.quantize_per_tensor.d

In [23]:
quantized_edge_model = litert_torch.convert(
    quantized_model,
    sample_inputs,
    quant_config=QuantConfig(
        pt2e_quantizer=quantizer
    ),
)

print("Quantized LiteRT conversion succeeded")

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default (+00:00)

... (15 outputs left out)

(00:01) [START] LiteRT-Torch Convert > Run LiteRT Converter Passes

(00:01) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:01) [ DONE] LiteRT-Torch Convert (+00:01)

Quantized LiteRT conversion succeeded


In [24]:
quantized_litert_output = quantized_edge_model(
    *sample_inputs
)

if isinstance(
    quantized_litert_output,
    (tuple, list),
):
    quantized_litert_output = (
        quantized_litert_output[0]
    )

quantized_litert_output = np.asarray(
    quantized_litert_output
)

pt2e_output_numpy = (
    quantized_torch_output
    .detach()
    .cpu()
    .numpy()
)

conversion_error = np.abs(
    pt2e_output_numpy
    - quantized_litert_output
)

print(
    "Quantized LiteRT output shape:",
    quantized_litert_output.shape,
)

print(
    "PT2E versus LiteRT maximum error:",
    conversion_error.max(),
)

print(
    "PT2E versus LiteRT mean error:",
    conversion_error.mean(),
)

(00:00) [START] Write Model to Bytes

I0000 00:00:1790154958.585392    9466 flatbuffer_export.cc:4342] Estimated count of arithmetic ops: 14.844 M  ops, equivalently 7.422 M  MACs


(00:00) [ DONE] Write Model to Bytes (+00:00)

Quantized LiteRT output shape: (1, 32, 128)
PT2E versus LiteRT maximum error: 0.0031100959
PT2E versus LiteRT mean error: 3.440884e-05


In [25]:
quantized_output_path = (
    output_directory
    / "tinystories_block0_w8a8.tflite"
)

quantized_edge_model.export(
    str(quantized_output_path)
)

print(
    "Saved quantized model:",
    quantized_output_path.resolve(),
)

print(
    "FP32 LiteRT size:",
    output_path.stat().st_size,
    "bytes",
)

print(
    "W8A8 LiteRT size:",
    quantized_output_path.stat().st_size,
    "bytes",
)

Saved quantized model: /home/randy/Documents/ACompArc/model/project/models/tinystories_block0_w8a8.tflite
FP32 LiteRT size: 818532 bytes
W8A8 LiteRT size: 232432 bytes
